In [ ]:
# Initial Setup for Notebook
import pathlib
import data_conduit
from data_conduit.qc import (
    qc_datastructure,
    plot_path_grid,
    DEFAULT_CENTROID_POINTS,
    training_spec,
    training_session_names,
    filter_trials,
    summarise_training_filter,
)

from movement.plots import plot_centroid_trajectory
import movement.kinematics as kin
from data_conduit.qc.slicing import slice_pose_for_trial
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import xarray as xr

# Training data root. The sessions listed in Training-trials_detail.xlsx live under
# BonsaiOutput/Training (NOT the Test tree). Their spreadsheet "day" numbers do not
# match the on-disk Day<N> folders, so sessions are selected by Bonsai filename via
# `include=` in the next cell, not by day.
ROOT = pathlib.Path('/media/sepi/Elements1/PathIntegrationProtocol/BonsaiOutput/Training')

In [2]:
# Session selection (step 1 of 2)
# Load ONLY the training sessions listed in Celia's spreadsheet, transcribed in
# data_conduit.qc.TRAINING_DETAIL. This yields the FULL, UNFILTERED trial table so it
# can be inspected here; trial-level filtering is done separately in the next cell.
spec = training_spec()                       # tidy per-session rules (min_trial, exclude_on, ...)

datastructure = qc_datastructure(
    root = ROOT,
    depth = 2,                                                                        # session folders sit 2 levels below root: <mouseID>/<day>/<session>
    level_names = ('mouseID', 'day'),                                                 # provenance columns kept on every entry
    streams = ('events', 'nosepoke', 'soundcard', 'session_settings', 'video', 'dlc'),
    include = training_session_names(spec),                                           # restrict to the spreadsheet's Bonsai sessions (matched by session name, NOT day)
    # exclude = None,
    # device_yaml = './device.yml',
    # soundcard_yaml = './soundcard.yml',
    # nosepoke_count = 18,
    # trial_start_buffer = 0.0,
    # l0_selector = None,                                                             # l0 -> mouseID, l1 -> day (l2/session filtering is done by include= above)
)

result = datastructure.load()                # apply the catalog to each session and concatenate across sessions
trials = result['trials']                    # unfiltered: every trial of every selected session (inspect before filtering)
trials


                  If any files/paths are missing or invalid for the monosource_data_arrays you specified, 
                  you may not see warnings about them. Set verbose=True to enable warnings.
                  

                  If any files/paths are missing or invalid for the monosource_data_arrays you specified, 
                  you may not see warnings about them. Set verbose=True to enable warnings.
                  

                  If any files/paths are missing or invalid for the monosource_data_arrays you specified, 
                  you may not see warnings about them. Set verbose=True to enable warnings.
                  

                  If any files/paths are missing or invalid for the monosource_data_arrays you specified, 
                  you may not see warnings about them. Set verbose=True to enable warnings.
                  


2026-07-03 17:58:32.883 | WARNING  | data_conduit.datastructure:read_session:236 - /home/sepi/dataconduit_workspace/data-conduit/src/data_conduit/datastructure.py:551: UserWarning: reader 'dlc' skipped for session '2026-05-28T154840Z': FileNotFoundError: no 'DLC' folder in session /media/sepi/Elements1/PathIntegrationProtocol/BonsaiOutput/Training/MbL_M01569517/Day17/2026-05-28T154840Z.
  objects = self.catalog.read_session(info['path'])

2026-07-03 17:58:33.074 | WARNING  | data_conduit.datastructure:read_session:236 - /home/sepi/dataconduit_workspace/data-conduit/src/data_conduit/datastructure.py:551: UserWarning: reader 'dlc' skipped for session '2026-05-29T155941Z': FileNotFoundError: no 'DLC' folder in session /media/sepi/Elements1/PathIntegrationProtocol/BonsaiOutput/Training/MbL_M01569517/Day18/2026-05-29T155941Z.
  objects = self.catalog.read_session(info['path'])



,trial_index,start_time,end_time,tz_triggered_time,outbound_start_time,outbound_end_time,inbound_start_time,inbound_end_time,TTT,TTP,ChosenPort,CorrectPort,outcome,LED,angle_offset,target_zone_size,session,mouseID,day
0,1,22852.592992,22879.349984,22874.800992,22852.592992,22874.800992,22874.800992,22879.349984,22.208000,4.548992,14,14,Success,ON,0.0,150.0,2026-04-23T160907Z,FLR_M01569521,Day2
1,2,22879.349984,22923.335488,22921.252000,22879.349984,22921.252000,22921.252000,22923.335488,41.902016,2.083488,14,14,Success,ON,0.0,150.0,2026-04-23T160907Z,FLR_M01569521,Day2
2,3,22923.335488,22935.201984,22932.852992,22923.335488,22932.852992,22932.852992,22935.201984,9.517504,2.348992,14,14,Success,ON,0.0,150.0,2026-04-23T160907Z,FLR_M01569521,Day2
3,4,22935.201984,22945.540000,22944.252000,22935.201984,22944.252000,22944.252000,22945.540000,9.050016,1.288000,14,14,Success,ON,0.0,150.0,2026-04-23T160907Z,FLR_M01569521,Day2
4,5,22945.540000,22955.582464,22954.052000,22945.540000,22954.052000,22954.052000,22955.582464,8.512000,1.530464,14,14,Success,ON,0.0,150.0,2026-04-23T160907Z,FLR_M01569521,Day2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2479,60,16173.513984,16196.815488,16191.792000,16173.513984,16191.792000,16191.792000,16196.815488,18.278016,5.023488,-1,16,Miss,OFF,NaN,150.0,2026-05-14T135225Z,MbR_M01569518,Day9
2480,61,16196.815488,16374.456992,16369.442976,16196.815488,16369.442976,16369.442976,16374.456992,172.627488,5.014016,-1,16,Miss,OFF,NaN,150.0,2026-05-14T135225Z,MbR_M01569518,Day9
2481,62,16374.456992,16461.362976,16456.342976,16374.456992,16456.342976,16456.342976,16461.362976,81.885984,5.020000,-1,16,Miss,OFF,NaN,150.0,2026-05-14T135225Z,MbR_M01569518,Day9
2482,63,16461.362976,16474.692000,16472.442976,16461.362976,16472.442976,16472.442976,16474.692000,11.080000,2.249024,16,16,Success,OFF,NaN,150.0,2026-05-14T135225Z,MbR_M01569518,Day9


In [3]:
print("Loaded streams:\n============================================")
for stream in datastructure:
    print(f'    ID| {stream}\n    ======================')
    display(datastructure[stream].to_dataframe().head() if hasattr(datastructure[stream], 'to_dataframe') else datastructure[stream].head())
    print(f'Total entries: {len(datastructure[stream])}\n\n')


Loaded streams:
    ID| nosepoke:Activations


,,device,register,localID,session,mouseID,day,Activations_data
Time,peripherals,,,,,,,


Total entries: 0


    ID| nosepoke:LEDs


,,device,register,localID,session,mouseID,day,LEDs_data
Time,peripherals,,,,,,,


Total entries: 0


    ID| nosepoke:Valves


...

Total entries: 991102


    ID| dlc:confidence


session        mouseID   day  confidence
Time         keypoints                                                     
22849.683584 nose       2026-04-23T160907Z  FLR_M01569521  Day2    0.978499
             lear       2026-04-23T160907Z  FLR_M01569521  Day2    0.931897
             rear       2026-04-23T160907Z  FLR_M01569521  Day2    0.829694
             body       2026-04-23T160907Z  FLR_M01569521  Day2    0.846847
             tailbase   2026-04-23T160907Z  FLR_M01569521  Day2    0.830134

Total entries: 991102


    ID| trials


,trial_index,start_time,end_time,tz_triggered_time,outbound_start_time,outbound_end_time,inbound_start_time,inbound_end_time,TTT,TTP,ChosenPort,CorrectPort,outcome,LED,angle_offset,target_zone_size,session,mouseID,day
0,1,22852.592992,22879.349984,22874.800992,22852.592992,22874.800992,22874.800992,22879.349984,22.208000,4.548992,14,14,Success,ON,0.0,150.0,2026-04-23T160907Z,FLR_M01569521,Day2
1,2,22879.349984,22923.335488,22921.252000,22879.349984,22921.252000,22921.252000,22923.335488,41.902016,2.083488,14,14,Success,ON,0.0,150.0,2026-04-23T160907Z,FLR_M01569521,Day2
2,3,22923.335488,22935.201984,22932.852992,22923.335488,22932.852992,22932.852992,22935.201984,9.517504,2.348992,14,14,Success,ON,0.0,150.0,2026-04-23T160907Z,FLR_M01569521,Day2
3,4,22935.201984,22945.540000,22944.252000,22935.201984,22944.252000,22944.252000,22945.540000,9.050016,1.288000,14,14,Success,ON,0.0,150.0,2026-04-23T160907Z,FLR_M01569521,Day2
4,5,22945.540000,22955.582464,22954.052000,22945.540000,22954.052000,22954.052000,22955.582464,8.512000,1.530464,14,14,Success,ON,0.0,150.0,2026-04-23T160907Z,FLR_M01569521,Day2


Total entries: 2484




In [4]:
datastructure['nosepoke:Activations']

<xarray.DataArray 'Activations_data' (Time: 0, peripherals: 18)> Size: 0B

Coordinates:
  * Time         (Time) float64 0B 
    session      (Time) <U18 0B 
    mouseID      (Time) object 0B 
    day          (Time) object 0B 
  * peripherals  (peripherals) <U5 360B 'NP_0' 'NP_1' 'NP_2' ... 'NP_16' 'NP_17'
    device       (peripherals) <U9 648B 'Behavior0' 'Behavior0' ... 'Behavior5'
    register     (peripherals) <U2 144B '32' '32' '32' '32' ... '32' '32' '32'
    localID      (peripherals) <U7 504B 'DIPort0' 'DIPort1' ... 'DIPort2'
Attributes:
    description:        Base DataArray for unified coordinates of type Activa...
    source:             Constructed using construct_base_da function
    _lookup_array_ref:  <xarray.DataArray 'Activations_lookup' (peripherals: ...

# Filtering

Trials are filtered per `Training-trials_detail.xlsx` (Celia), transcribed into
`data_conduit.qc.TRAINING_DETAIL`. This is split into **two independent steps** so the
unfiltered trial table can be inspected first:

1. **Session selection** (cell above): `training_session_names(spec)` restricts the load to
   exactly the Bonsai sessions the spreadsheet lists. Sessions are matched by their **Bonsai
   filename, not by day** — the spreadsheet's day numbers do *not* line up with the on-disk
   `Day<N>` folders (e.g. FL "day 2" = `2026-05-11T125643Z` actually lives under `Day7`).

2. **Trial filtering** (cell below): `filter_trials(trials, spec)` applies the per-session rules:
   - every session: keep `trial_index >= min_trial` (the spreadsheet's *"X to end"*);
   - `MR_M01569515` / `MbR_M01569518` only: additionally drop the mid-session cue trials — the
     block protocol runs `1 ON, then 20/25 OFF`, so the cue trials are exactly the `LED == 'ON'`
     trials in the kept range. **All** in-range ON trials are dropped, which follows the recorded
     LED state even where a session ran short or over-ran the spreadsheet's annotated block count.

   Combined days (the *"COMBINE ... SESSIONS"* rows) share one `training_day` and are renumbered
   `1..N` across their pooled sessions as `day_trial_index`. Edit `TRAINING_DETAIL` in
   `training_filter.py` if the spreadsheet changes.

In [5]:
# Trial filtering (step 2 of 2)
# Apply the per-session spreadsheet rules to the loaded `trials`. Kept separate from
# session selection so the unfiltered `trials` can be inspected above. `filter_trials`
# prints a per-session keep/drop summary (report=True by default); pass report=False to
# silence it, or call summarise_training_filter(trials, spec) to preview without filtering.
trials_filtered = filter_trials(trials, spec)
trials_filtered

      mouseID  training_day            session  n_total  min_trial  n_after_cut  exclude_on  n_on_dropped  n_kept
FLR_M01569521             1 2026-04-23T160907Z       74         11           64       False             0      64
FLR_M01569521             2 2026-04-25T190120Z       22         11           12       False             0      12
FLR_M01569521             2 2026-04-25T190915Z       73         11           63       False             0      63
FLR_M01569521             3 2026-04-28T124013Z       14          6            9       False             0       9
FLR_M01569521             3 2026-04-28T124547Z       68         11           58       False             0      58
FLR_M01569521             4 2026-04-29T161826Z      131         11          121       False             0     121
FLR_M01569521             5 2026-04-30T151438Z      109          6          104       False             0     104
 FL_M01569519             2 2026-05-11T125643Z       98         16           83       Fa

,trial_index,start_time,end_time,tz_triggered_time,outbound_start_time,outbound_end_time,inbound_start_time,inbound_end_time,TTT,TTP,...,CorrectPort,outcome,LED,angle_offset,target_zone_size,session,mouseID,day,training_day,day_trial_index
0,11,23009.675488,23021.869984,23016.854976,23009.675488,23016.854976,23016.854976,23021.869984,7.179488,5.015008,...,14,Miss,OFF,0.0,150.0,2026-04-23T160907Z,FLR_M01569521,Day2,1,1
1,12,23021.869984,23030.656000,23027.952992,23021.869984,23027.952992,23027.952992,23030.656000,6.083008,2.703008,...,14,Success,OFF,NaN,150.0,2026-04-23T160907Z,FLR_M01569521,Day2,1,2
2,13,23030.656000,23040.496480,23037.102976,23030.656000,23037.102976,23037.102976,23040.496480,6.446976,3.393504,...,14,Success,OFF,0.0,150.0,2026-04-23T160907Z,FLR_M01569521,Day2,1,3
3,14,23040.496480,23051.965984,23046.953984,23040.496480,23046.953984,23046.953984,23051.965984,6.457504,5.012000,...,14,Miss,OFF,0.0,150.0,2026-04-23T160907Z,FLR_M01569521,Day2,1,4
4,15,23051.965984,23066.218976,23061.202976,23051.965984,23061.202976,23061.202976,23066.218976,9.236992,5.016000,...,14,Miss,OFF,NaN,150.0,2026-04-23T160907Z,FLR_M01569521,Day2,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1855,88,8522.236992,8533.254496,8529.316992,8522.236992,8529.316992,8529.316992,8533.254496,7.080000,3.937504,...,13,Success,OFF,NaN,150.0,2026-05-20T102711Z,MbR_M01569518,Day13,5,62
1856,89,8533.254496,8552.430496,8547.418976,8533.254496,8547.418976,8547.418976,8552.430496,14.164480,5.011520,...,13,Miss,OFF,0.0,150.0,2026-05-20T102711Z,MbR_M01569518,Day13,5,63
1857,90,8552.430496,8565.830496,8560.817984,8552.430496,8560.817984,8560.817984,8565.830496,8.387488,5.012512,...,13,Miss,OFF,NaN,150.0,2026-05-20T102711Z,MbR_M01569518,Day13,5,64
1858,91,8565.830496,8592.507488,8587.717984,8565.830496,8587.717984,8587.717984,8592.507488,21.887488,4.789504,...,13,Success,OFF,NaN,150.0,2026-05-20T102711Z,MbR_M01569518,Day13,5,65


In [ ]:
# Figure 1 | Outbound & inbound trajectories
# Grid: rows = training day, columns = each mouse split into outbound|inbound
# subplots. Trajectories are the DLC centroid path per trial, overlaid on the arena
# frame and coloured light->dark by trial order. Needs DLC pose (sorted into each
# session's DLC/ folder) + the `movement` package; a session with no pose renders as
# the bare frame.
#
# IMPORTANT: DLC was run on the UNDISTORTED video (1620x1260), so the pose lives in
# undistorted pixel coordinates. The background must therefore be the
# `UndistortedVideoData` frame, NOT the raw `VideoData` frame (1440x1080) - drawing
# the raw frame leaves the scatter off-centre / mis-scaled relative to the arena.
#
# The colourbar is labelled with `day_trial_index` (the contiguous 1..N per-day
# counter from filter_trials) so its range matches each panel's trial count `n`;
# the raw `trial_index` restarts and gaps across a combined day, so it would not.
def figure1(result, trials, root, *, mice=None, groups=None,
            row_by='training_day', col_by=('mouseID', 'segment'),
            segments=('outbound', 'inbound'), video_subdir='UndistortedVideoData',
            trial_number_column='day_trial_index', **kwargs):
    """Draw the trajectory grid (via plot_path_grid) for the FILTERED trials.

    mice         : subset of mouseIDs to plot (None = all present).
    groups       : {name: [mice]} to colour by group instead of per-mouse.
    video_subdir : session subfolder for the background frame. Default
                   'UndistortedVideoData' to match the DLC coordinate space.
    trial_number_column : column labelling the colourbar ticks. Default
                   'day_trial_index' (contiguous per day, so it matches `n`).
    kwargs       : forwarded to plot_path_grid (centroid_points, marker_size,
                   cmap_range, figsize_per_cell, ...).
    """
    return plot_path_grid(
        {**result, 'trials': trials}, root,
        mice=mice, groups=groups,
        row_by=row_by, col_by=list(col_by), segments=segments,
        video_subdir=video_subdir,
        trial_number_column=trial_number_column,
        require_pose_for_trials=False,   # sessions/trials without pose -> blank, don't raise
        **kwargs,
    )


# One mouse at a time keeps the grid readable; pass mice=None (or a list) for more.
fig, axes = figure1(result, trials_filtered, ROOT, mice=['FbR_M01569522'])

# Figure 2 | Path lengths (outbound & inbound)

Path length = summed frame-to-frame distance of the **centroid** (mean of the DLC keypoints)
over a trial's segment, in pixels, computed with **`movement.kinematics.compute_path_length`**.
Low-likelihood frames are blanked first (`min_confidence`) so DLC jitter doesn't inflate the
distance; `nan_policy` controls how movement bridges the blanked frames. Steps:

1. `add_path_lengths(trials_filtered, result)` → adds `outbound_path_length` / `inbound_path_length`.
2. `figure2_distributions(...)` → per-session distributions (outbound | inbound as two columns).
3. `binned_relationship(...)` → the binned **scatter** relationships (path length on the y-axis),
   laid out as a grid: **one row per training day**, columns **2.1** path length vs success rate,
   **2.2** TTT vs success rate, **2.3** path length vs TTT. Each panel scatters every mouse
   separately (stable colour per mouse) and overlays **one pooled linear fit** across all mice's
   points (dashed, annotated with Pearson `r`).

`GROUPS`, `BINS`, and `binning` are the tunable knobs; set `regression=False` to drop the fit line.

In [ ]:
# Per-trial path length (centroid = mean of the DLC keypoints), via `movement`.
CENTROID = DEFAULT_CENTROID_POINTS   # ('nose', 'lear', 'rear', 'body', 'tailbase')


def _segment_path_length(centroid, row, segment, nan_policy):
    """movement path length of the centroid over one trial's segment window (px)."""
    sl = slice_pose_for_trial(centroid, row, segment=segment)
    if sl.sizes.get('Time', 0) < 2:
        return np.nan
    # movement.kinematics wants a `time` dim; it sums the displacement norms.
    return float(kin.compute_path_length(sl.rename({'Time': 'time'}),
                                         nan_policy=nan_policy, nan_warn_threshold=1.0))


def add_path_lengths(trials, result, *, segments=('outbound', 'inbound'),
                     centroid_points=CENTROID, min_confidence=0.6, nan_policy='ffill',
                     pose_key='dlc:position', confidence_key='dlc:confidence'):
    """Return `trials` with `<segment>_path_length` columns (px), computed with
    `movement.kinematics.compute_path_length`. NaN where a trial has no pose.
    `min_confidence` blanks jittery low-likelihood frames before summing (None = off);
    `nan_policy` ('ffill' | 'scale') is how movement bridges those blanked frames."""
    kp = list(centroid_points)
    centroid = result[pose_key].sel(keypoints=kp).mean('keypoints')          # (Time, space)
    if min_confidence is not None:
        mean_conf = result[confidence_key].sel(keypoints=kp).mean('keypoints')
        centroid = centroid.where(mean_conf >= min_confidence)               # low-conf frames -> NaN
    out = trials.copy()
    for seg in segments:
        out[f'{seg}_path_length'] = [
            _segment_path_length(centroid, row, seg, nan_policy) for _, row in out.iterrows()
        ]
    return out


trials_len = add_path_lengths(trials_filtered, result)   # adds outbound/inbound_path_length
trials_len[['mouseID', 'training_day', 'session',
            'outbound_path_length', 'inbound_path_length']].head()

In [ ]:
# Figure 2 (main) | Distribution of outbound & inbound path lengths, per session.
def figure2_distributions(trials, mouse, *, by='session', bins=30,
                          segments=('outbound', 'inbound'), mouse_column='mouseID'):
    """Path-length histograms for one mouse: one row per `by` value ('session' or
    'training_day'), one COLUMN per segment (outbound | inbound). The x-axis is shared
    down each column so days are comparable within a segment. `bins` sets the bins."""
    sub = trials[trials[mouse_column] == mouse]
    keys = list(pd.unique(sub[by]))
    ncols = len(segments)
    fig, axes = plt.subplots(len(keys), ncols, figsize=(4.6 * ncols, 2.2 * len(keys)),
                             squeeze=False, sharex='col')
    for i, key in enumerate(keys):
        cell = sub[sub[by] == key]
        for j, seg in enumerate(segments):
            ax = axes[i, j]
            vals = cell[f'{seg}_path_length'].dropna()
            if len(vals):
                ax.hist(vals, bins=bins, alpha=0.85, label=f'n={len(vals)}')
                ax.legend(fontsize=8)
            if j == 0:
                ax.set_ylabel(f'{by}={key}')
            if i == 0:
                ax.set_title(seg)
    for j in range(ncols):
        axes[-1, j].set_xlabel('centroid path length (px)')
    fig.suptitle(f'{mouse}: path-length distributions', y=1.0)
    fig.tight_layout()
    return fig, axes


# One figure per mouse (rows = training day, columns = outbound | inbound).
for mouse in pd.unique(trials_len['mouseID']):
    fig, axes = figure2_distributions(trials_len, mouse, by='training_day', bins=40)

In [ ]:
# Figure 2.1 - 2.3 | Binned relationships, per DAY (rows) x relationship (columns).
# Each cell scatters all mice separately (stable colour per mouse) for that day, plus
# a single pooled linear-regression line across ALL mice's points (dashed, with r).
def binned_relationship(trials, *, bin_by, x, y, groups=None, bins=8, binning='equal',
                        ax=None, mouse_column='mouseID', regression=False, legend=True,
                        colors=None):
    """Bin trials by `bin_by` into `bins`, then SCATTER each (group) series as the
    per-bin aggregate `x` vs `y` (one point per bin). With regression=True, fit ONE
    line across every group's points (pooled) and annotate the Pearson r.

    x / y : column name (per-bin mean) or 'success_rate' (fraction outcome=='Success').
    groups: {name: [mice]} -> one series per group.
    colors: {group: colour} for stable colours across panels.
    binning: 'equal' (equal-width) or 'quantile' (equal-count, robust to skew).
    """
    d = trials.copy()
    d['_success'] = (d['outcome'] == 'Success').astype(float)
    col = lambda name: '_success' if name == 'success_rate' else name
    if groups is not None:
        d['_group'] = d[mouse_column].map({m: g for g, mice in groups.items() for m in mice})
        d = d[d['_group'].notna()]
    else:
        d['_group'] = 'all'
    d = d.dropna(subset=[bin_by, col(x), col(y)])
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 4.5))
    xs_all, ys_all = [], []
    for group, gd in d.groupby('_group'):
        cut = (pd.qcut(gd[bin_by], bins, duplicates='drop') if binning == 'quantile'
               else pd.cut(gd[bin_by], bins))
        agg = gd.groupby(cut, observed=True).agg(xv=(col(x), 'mean'), yv=(col(y), 'mean'))
        ax.scatter(agg['xv'], agg['yv'], s=18, label=group, color=(colors or {}).get(group))
        xs_all.append(agg['xv'].values)
        ys_all.append(agg['yv'].values)
    if regression and xs_all:
        xp, yp = np.concatenate(xs_all), np.concatenate(ys_all)
        m = np.isfinite(xp) & np.isfinite(yp)
        if m.sum() >= 2:
            slope, intercept = np.polyfit(xp[m], yp[m], 1)
            edge = np.array([xp[m].min(), xp[m].max()])
            ax.plot(edge, slope * edge + intercept, 'k--', lw=1.5)
            ax.text(0.03, 0.96, f'r={np.corrcoef(xp[m], yp[m])[0, 1]:.2f}', transform=ax.transAxes,
                    va='top', fontsize=9, bbox=dict(boxstyle='round', fc='white', alpha=0.6, ec='none'))
    ax.set_xlabel('success rate' if x == 'success_rate' else x)
    ax.set_ylabel('success rate' if y == 'success_rate' else y)
    if legend:
        ax.legend(fontsize=7, title='mouse')
    return ax


from matplotlib.lines import Line2D

# One series per mouse, with a stable colour per mouse across every panel.
GROUPS = {mouse: [mouse] for mouse in sorted(pd.unique(trials_len['mouseID']))}
COLORS = {group: f'C{i % 10}' for i, group in enumerate(GROUPS)}
BINS = 8   # fewer bins than the pooled view: each mouse-day has far fewer trials

# The three relationships (path length on the y-axis for 2.1 / 2.3).
RELATIONSHIPS = [
    dict(bin_by='outbound_path_length', x='success_rate', y='outbound_path_length',
         title='2.1  path length vs success rate'),
    dict(bin_by='TTT', x='success_rate', y='TTT',
         title='2.2  TTT vs success rate'),
    dict(bin_by='outbound_path_length', x='TTT', y='outbound_path_length',
         title='2.3  path length vs TTT'),
]

# Grid: one ROW per training day, one COLUMN per relationship.
days = sorted(pd.unique(trials_len['training_day']))
fig, axes = plt.subplots(len(days), len(RELATIONSHIPS),
                         figsize=(5 * len(RELATIONSHIPS), 3.4 * len(days)), squeeze=False)
for i, day in enumerate(days):
    day_trials = trials_len[trials_len['training_day'] == day]
    for j, rel in enumerate(RELATIONSHIPS):
        binned_relationship(day_trials, bin_by=rel['bin_by'], x=rel['x'], y=rel['y'],
                            groups=GROUPS, bins=BINS, binning='quantile',
                            ax=axes[i, j], regression=True, legend=False, colors=COLORS)
        if i == 0:
            axes[i, j].set_title(rel['title'])
    axes[i, 0].set_ylabel(f"day {day}\n" + axes[i, 0].get_ylabel())

legend_handles = [Line2D([0], [0], marker='o', ls='', color=COLORS[g], label=g) for g in GROUPS]
legend_handles.append(Line2D([0], [0], ls='--', color='k', label='linear fit (all mice)'))
fig.legend(handles=legend_handles, loc='upper center', ncol=len(legend_handles),
           fontsize=8, bbox_to_anchor=(0.5, 1.0))
fig.tight_layout(rect=[0, 0, 1, 0.985])